<a href="https://colab.research.google.com/github/VasilisPapageorgiou/Bayesian-Embeddings/blob/main/Sim1_Bayesian_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# SIMULATION 1 — THEORY-CONSISTENT PILOT V2 (MIXING-CALIBRATED)
# ============================================================
#
# Statistical objective:
#   1. Structural allocation recovery.
#   2. Posterior spectral/tail estimation.
#   3. Adaptive budget selection.
#   4. Predictive efficiency and parameter reduction.
#
# Design:
#   spectral regime = {clear, weak}
#   n_train         = {500, 1000, 2000}
#
# Bayesian stage:
#   Implements the rank–spectral prior used in Section 2.3:
#     r_j ~ Uniform{0,...,K_j};
#     U_j and V_j have Haar-invariant orientation priors;
#     s_{j1}^2 ~ Gamma(a_s,b_s);
#     s_{j,k+1}^2 = rho_{jk}s_{jk}^2,
#     rho_{jk} ~ Beta(1,1).
#
# Posterior computation:
#   Metropolis-within-Gibbs implementing Remark 2.3 literally.
#   Conditional on r_j, unconstrained Gaussian auxiliary matrices
#   A_j and B_j are mapped to U_j and V_j through QR decompositions.
#   pCN proposals preserve the N(0,1) auxiliary priors exactly;
#   transformed random walks update the singular-value parameters.
#   Adjacent reversible-jump birth/death moves update r_j, with new
#   auxiliary columns and singular-value variables proposed from their
#   conditional priors, so the induced marginal prior is exactly the
#   rank–spectral prior of Section 2.3.
#
# Dynamic programming:
#   exact Multiple-Choice Knapsack DP.
#
# GOOGLE COLAB — ONE CELL
#
# At startup:
#   1 = start from beginning
#   2 = resume from latest compatible checkpoint
#
# Checkpoints:
#   - after every completed Monte Carlo replication
#   - additionally approximately every 20 minutes
# ============================================================


# ============================================================
# 0. START / RESUME
# ============================================================

MODE = input(
    "Simulation 1 theory-consistent: type 1 to START FROM THE BEGINNING "
    "or 2 to RESUME FROM THE LAST CHECKPOINT: "
).strip()

if MODE not in {"1", "2"}:
    raise ValueError(
        "Please run the cell again and enter either 1 or 2."
    )


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import gc
import json
import math
import time
import pickle
import random
import hashlib
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

DEVICE = torch.device("cpu")

torch.set_num_threads(
    max(
        1,
        min(
            4,
            os.cpu_count() or 1
        )
    )
)


# ============================================================
# 2. GOOGLE DRIVE
# ============================================================

try:

    from google.colab import drive

    drive.mount(
        "/content/drive"
    )

    ROOT = (
        "/content/drive/MyDrive/"
        "JRSSB_Embedding_Allocation/"
        "Simulation1_theory_consistent_pilot_v2"
    )

except Exception:

    ROOT = (
        "./JRSSB_Embedding_Allocation/"
        "Simulation1_theory_consistent_pilot_v2"
    )

    print(
        "Google Drive was not detected. "
        "Using local directory:",
        ROOT
    )

os.makedirs(
    ROOT,
    exist_ok=True
)

CHECKPOINT_PATH = os.path.join(
    ROOT,
    "simulation1_theory_consistent_pilot_v2_checkpoint.pkl"
)

RESULTS_PATH = os.path.join(
    ROOT,
    "simulation1_theory_consistent_pilot_v2_replication_results.csv"
)

BUDGET_RESULTS_PATH = os.path.join(
    ROOT,
    "simulation1_theory_consistent_pilot_v2_budget_results.csv"
)

SUMMARY_PATH = os.path.join(
    ROOT,
    "simulation1_theory_consistent_pilot_v2_summary.csv"
)


# ============================================================
# 3. SIMULATION CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# False runs the final 100-replication experiment.
# True can be used for a shorter diagnostic run.
# ------------------------------------------------------------

PILOT_MODE = False

BASE_SEED = 271828

CHECKPOINT_SECONDS = 20 * 60


# ------------------------------------------------------------
# Categorical predictors
# ------------------------------------------------------------

N_LEVELS = [
    8,
    10,
    12,
    9
]

M = len(
    N_LEVELS
)


# ------------------------------------------------------------
# Latent and response dimensions
# ------------------------------------------------------------

P_Z = 4

Q = 4

P_U = 2

K_LIST = [
    min(
        nj - 1,
        P_Z
    )
    for nj in N_LEVELS
]


# ------------------------------------------------------------
# Validation / testing
# ------------------------------------------------------------

N_VAL = 500

N_TEST = 3000


# ------------------------------------------------------------
# Adaptive-budget tolerance
# ------------------------------------------------------------

ZETA_N = 0.0015


# ------------------------------------------------------------
# Bayesian rank–spectral prior: Section 2.3
# ------------------------------------------------------------

# p_j(k)>0 for k=0,...,K_j.  A uniform prior is used here.
RANK_PRIOR = "uniform"

# s_{j1}^2 ~ Gamma(shape, rate)
SINGULAR_SHAPE = 2.0
SINGULAR_RATE = 1.0

# rho_{jk} ~ Beta(1,1)
RATIO_ALPHA = 1.0
RATIO_BETA = 1.0

TAU_GAMMA = 2.5
TAU_MU = 2.5

SIGMA_A0 = 2.5
SIGMA_B0 = 0.02


# ------------------------------------------------------------
# Reduced MLP
# ------------------------------------------------------------

# Section 2.4 requires the same hidden-layer architecture as the
# reference MLP, with only the input dimension changed.  The reference
# model has one hidden layer of width P_Z.
RED_HIDDEN = (
    P_Z,
)

RED_WEIGHT_DECAY = 5e-4


# ------------------------------------------------------------
# Training / MCMC settings
# ------------------------------------------------------------

if PILOT_MODE:

    N_REPS = 10

    REF_EPOCHS = 300

    RED_EPOCHS = 250

    REF_RESTARTS = 5

    RED_RESTARTS = 5

    # Diagnostic pilot: enough post-burn-in draws to inspect posterior
    # uncertainty and mixing before the 100-replication experiment.
    MCMC_ITERS = 2000

    MCMC_BURN = 1000

    MCMC_THIN = 4

else:

    N_REPS = 100

    REF_EPOCHS = 400

    RED_EPOCHS = 300

    REF_RESTARTS = 5

    RED_RESTARTS = 5

    MCMC_ITERS = 1500

    MCMC_BURN = 750

    MCMC_THIN = 5


# ------------------------------------------------------------
# MCMC tuning constants
# ------------------------------------------------------------

MCMC_ADAPT_EVERY = 25
MCMC_ADAPT_FRACTION = 0.80

# The orientation and spectral coordinates are updated in separate
# Metropolis blocks.  This avoids the poor mixing caused by proposing
# a simultaneous change in U_j, V_j and the complete spectrum.
MCMC_TARGET_ORIENTATION = 0.30
MCMC_TARGET_SPECTRUM = 0.30
MCMC_TARGET_GAMMA = 0.25
MCMC_TARGET_MU = 0.30

# pCN coefficient for the Gaussian auxiliary matrices A_j and B_j.
AUXILIARY_INITIAL_BETA = 0.10

# Random-walk scale on one transformed spectral coordinate at a time
# (log s_{j1}^2 or one logit rho_{jk}).
SPECTRUM_INITIAL_STEP = 0.08

# Attempt an adjacent birth/death move frequently enough to diagnose
# rank mixing in the short pilot.
RANK_MOVE_EVERY = 2

# Gamma and mu random-walk proposals.
GAMMA_INITIAL_STEP = 0.010
MU_INITIAL_STEP = 0.010
MCMC_REFERENCE_N = 500.0

# Robbins--Monro multiplicative adaptation intensity.
MCMC_ADAPT_GAIN = 0.70

AUXILIARY_BETA_MIN = 0.01
AUXILIARY_BETA_MAX = 0.70

SPECTRUM_STEP_MIN = 0.002
SPECTRUM_STEP_MAX = 0.300

GAMMA_STEP_MIN = 0.0005
GAMMA_STEP_MAX = 0.050

MU_STEP_MIN = 0.0005
MU_STEP_MAX = 0.050

# ------------------------------------------------------------
# Clean factorial design
# ------------------------------------------------------------

NOISE_SD = 0.10

SCENARIOS = []

for spectral_regime in [
    "clear",
    "weak"
]:

    for n_train in [
        500,
        1000,
        2000
    ]:

        SCENARIOS.append(
            {
                "name":
                    f"{spectral_regime}_n{n_train}",

                "n_train":
                    n_train,

                "spectral_regime":
                    spectral_regime,

                "noise_sd":
                    NOISE_SD
            }
        )


# ============================================================
# 4. BUDGET GRID
# ============================================================

MIN_COST = sum(
    N_LEVELS
)

MAX_COST = sum(
    N_LEVELS[j]
    * K_LIST[j]
    for j in range(M)
)

BUDGETS = [
    MIN_COST,
    60,
    80,
    100,
    120,
    140,
    MAX_COST
]


# ============================================================
# 5. CONFIGURATION HASH
# ============================================================

CONFIG_FOR_HASH = {
    "pilot": PILOT_MODE,
    "n_reps": N_REPS,
    "n_levels": N_LEVELS,
    "p_z": P_Z,
    "q": Q,
    "p_u": P_U,
    "n_val": N_VAL,
    "n_test": N_TEST,
    "noise_sd": NOISE_SD,
    "budgets": BUDGETS,
    "zeta_n": ZETA_N,
    "rank_prior": RANK_PRIOR,
    "singular_shape": SINGULAR_SHAPE,
    "singular_rate": SINGULAR_RATE,
    "ratio_alpha": RATIO_ALPHA,
    "ratio_beta": RATIO_BETA,
    "tau_gamma": TAU_GAMMA,
    "tau_mu": TAU_MU,
    "sigma_a0": SIGMA_A0,
    "sigma_b0": SIGMA_B0,
    "red_hidden": RED_HIDDEN,
    "red_weight_decay": RED_WEIGHT_DECAY,
    "ref_epochs": REF_EPOCHS,
    "red_epochs": RED_EPOCHS,
    "ref_restarts": REF_RESTARTS,
    "red_restarts": RED_RESTARTS,
    "mcmc_iters": MCMC_ITERS,
    "mcmc_burn": MCMC_BURN,
    "mcmc_thin": MCMC_THIN,
    "adapt_every": MCMC_ADAPT_EVERY,
    "adapt_fraction": MCMC_ADAPT_FRACTION,
    "target_orientation": MCMC_TARGET_ORIENTATION,
    "target_spectrum": MCMC_TARGET_SPECTRUM,
    "target_gamma": MCMC_TARGET_GAMMA,
    "target_mu": MCMC_TARGET_MU,
    "auxiliary_initial_beta": AUXILIARY_INITIAL_BETA,
    "spectrum_initial_step": SPECTRUM_INITIAL_STEP,
    "rank_move_every": RANK_MOVE_EVERY,
    "gamma_initial_step": GAMMA_INITIAL_STEP,
    "mu_initial_step": MU_INITIAL_STEP,
    "adapt_gain": MCMC_ADAPT_GAIN,
    "auxiliary_beta_min": AUXILIARY_BETA_MIN,
    "auxiliary_beta_max": AUXILIARY_BETA_MAX,
    "spectrum_step_min": SPECTRUM_STEP_MIN,
    "spectrum_step_max": SPECTRUM_STEP_MAX,
    "gamma_step_min": GAMMA_STEP_MIN,
    "gamma_step_max": GAMMA_STEP_MAX,
    "mu_step_min": MU_STEP_MIN,
    "mu_step_max": MU_STEP_MAX,
    "scenarios": SCENARIOS
}

CONFIG_SIGNATURE = hashlib.md5(
    json.dumps(
        CONFIG_FOR_HASH,
        sort_keys=True
    ).encode()
).hexdigest()


# ============================================================
# 6. REPRODUCIBILITY
# ============================================================

def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)


# ============================================================
# 7. BASIC LINEAR-ALGEBRA HELPERS
# ============================================================

def random_orthogonal(
    d,
    rng
):

    A = rng.normal(
        size=(
            d,
            d
        )
    )

    Qm, R = np.linalg.qr(
        A
    )

    signs = np.sign(
        np.diag(
            R
        )
    )

    signs[
        signs == 0
    ] = 1.0

    Qm = (
        Qm
        * signs
    )

    return Qm


def centered_orthonormal_matrix(
    n,
    k,
    rng
):

    A = rng.normal(
        size=(
            n,
            k
        )
    )

    A -= A.mean(
        axis=0,
        keepdims=True
    )

    Qm, _ = np.linalg.qr(
        A
    )

    Qm -= Qm.mean(
        axis=0,
        keepdims=True
    )

    Qm, _ = np.linalg.qr(
        Qm
    )

    return Qm[
        :,
        :k
    ]


def make_theta_from_singular_values(
    n,
    pz,
    singular_values,
    rng
):

    k = min(
        n - 1,
        pz
    )

    U = centered_orthonormal_matrix(
        n,
        k,
        rng
    )

    V = random_orthogonal(
        pz,
        rng
    )[
        :,
        :k
    ]

    s = np.asarray(
        singular_values[
            :k
        ],
        dtype=float
    )

    theta = (
        U
        @ np.diag(
            s
        )
        @ V.T
    )

    theta -= theta.mean(
        axis=0,
        keepdims=True
    )

    return theta


# ============================================================
# 8. TRUE SPECTRAL STRUCTURE
# ============================================================

def get_spectral_values(
    regime
):

    if regime == "clear":

        spectra = [

            [
                1.80,
                0.30,
                0.10,
                0.05
            ],

            [
                1.60,
                0.70,
                0.20,
                0.05
            ],

            [
                2.00,
                1.00,
                0.40,
                0.10
            ],

            [
                1.50,
                0.50,
                0.20,
                0.05
            ]
        ]

    elif regime == "weak":

        spectra = [

            [
                1.80,
                1.20,
                0.90,
                0.60
            ],

            [
                1.60,
                1.30,
                1.00,
                0.70
            ],

            [
                2.00,
                1.60,
                1.30,
                1.00
            ],

            [
                1.50,
                1.10,
                0.85,
                0.60
            ]
        ]

    else:

        raise ValueError(
            "Unknown spectral regime."
        )

    return [
        np.asarray(
            spectra[j][
                :K_LIST[j]
            ],
            dtype=float
        )
        for j in range(M)
    ]


# ============================================================
# 9. TRUE INJECTIVE REFERENCE MLP
# ============================================================

def make_teacher_weights(
    seed=8877
):

    rng = np.random.default_rng(
        seed
    )

    W1 = random_orthogonal(
        P_Z,
        rng
    )

    W2 = random_orthogonal(
        P_Z,
        rng
    )

    b1 = np.array(
        [
            0.10,
            -0.10,
            0.05,
            -0.05
        ],
        dtype=float
    )

    b2 = np.array(
        [
            0.05,
            -0.05,
            0.05,
            -0.05
        ],
        dtype=float
    )

    return [
        (
            W1,
            b1
        ),
        (
            W2,
            b2
        )
    ]


TEACHER_WEIGHTS = (
    make_teacher_weights()
)


def mlp_forward_numpy(
    z,
    weights
):

    W1, b1 = weights[
        0
    ]

    W2, b2 = weights[
        1
    ]

    h = np.tanh(
        z
        @ W1.T
        +
        b1
    )

    return (
        h
        @ W2.T
        +
        b2
    )


# ============================================================
# 10. TRUE PARAMETERS
# ============================================================

def build_true_parameters(
    spectral_regime
):

    rng = np.random.default_rng(
        12345
    )

    singular_values = (
        get_spectral_values(
            spectral_regime
        )
    )

    theta_true = []

    for j in range(M):

        theta_true.append(
            make_theta_from_singular_values(
                N_LEVELS[j],
                P_Z,
                singular_values[j],
                rng
            )
        )

    gamma_true = rng.normal(
        scale=0.20,
        size=(
            P_U,
            P_Z
        )
    )

    mu_true = np.array(
        [
            0.10,
            -0.10,
            0.05,
            -0.05
        ]
    )

    return {

        "theta":
            theta_true,

        "gamma":
            gamma_true,

        "mu":
            mu_true,

        "singular_values":
            singular_values
    }


# ============================================================
# 11. DATA GENERATION
# ============================================================

def generate_dataset(
    n,
    true_par,
    noise_sd,
    seed
):

    rng = np.random.default_rng(
        seed
    )

    C = [

        rng.integers(
            0,
            N_LEVELS[j],
            size=n
        )

        for j in range(M)
    ]

    U = rng.normal(
        size=(
            n,
            P_U
        )
    )

    z = np.tile(
        true_par[
            "mu"
        ],
        (
            n,
            1
        )
    )

    z += (
        U
        @ true_par[
            "gamma"
        ]
    )

    for j in range(M):

        z += (
            true_par[
                "theta"
            ][j][
                C[j],
                :
            ]
        )

    y_mean = mlp_forward_numpy(
        z,
        TEACHER_WEIGHTS
    )

    y = (
        y_mean
        +
        noise_sd
        * rng.normal(
            size=y_mean.shape
        )
    )

    return (
        C,
        U,
        y,
        y_mean
    )


# ============================================================
# 12. REFERENCE MLP
# ============================================================

class ReferenceModel(
    nn.Module
):

    def __init__(
        self,
        seed=0
    ):

        super().__init__()

        set_seed(
            seed
        )

        self.embeddings = (
            nn.ModuleList(
                [
                    nn.Embedding(
                        N_LEVELS[j],
                        P_Z
                    )
                    for j in range(M)
                ]
            )
        )

        for emb in self.embeddings:

            nn.init.normal_(
                emb.weight,
                std=0.10
            )

        self.gamma = nn.Parameter(
            0.10
            * torch.randn(
                P_U,
                P_Z
            )
        )

        self.mu = nn.Parameter(
            torch.zeros(
                P_Z
            )
        )

        self.lin1 = nn.Linear(
            P_Z,
            P_Z
        )

        self.lin2 = nn.Linear(
            P_Z,
            P_Z
        )

        with torch.no_grad():

            q1, _ = torch.linalg.qr(
                torch.randn(
                    P_Z,
                    P_Z
                )
            )

            q2, _ = torch.linalg.qr(
                torch.randn(
                    P_Z,
                    P_Z
                )
            )

            self.lin1.weight.copy_(
                q1
            )

            self.lin2.weight.copy_(
                q2
            )

            self.lin1.bias.zero_()

            self.lin2.bias.zero_()

    def latent(
        self,
        C,
        U
    ):

        z = (
            self.mu.unsqueeze(
                0
            )
            +
            U
            @ self.gamma
        )

        for j in range(M):

            z = (
                z
                +
                self.embeddings[j](
                    C[j]
                )
            )

        return z

    def phi(
        self,
        z
    ):

        return self.lin2(
            torch.tanh(
                self.lin1(
                    z
                )
            )
        )

    def forward(
        self,
        C,
        U
    ):

        return self.phi(
            self.latent(
                C,
                U
            )
        )

    @torch.no_grad()
    def center_embeddings_(
        self
    ):

        for emb in self.embeddings:

            emb.weight -= (
                emb.weight.mean(
                    dim=0,
                    keepdim=True
                )
            )

    @torch.no_grad()
    def project_mlp_orthogonal_(
        self
    ):

        for layer in [
            self.lin1,
            self.lin2
        ]:

            U, _, Vh = (
                torch.linalg.svd(
                    layer.weight,
                    full_matrices=False
                )
            )

            layer.weight.copy_(
                U
                @ Vh
            )


# ============================================================
# 13. REFERENCE MODEL TRAINING
# ============================================================

def train_reference_once(
    C,
    U,
    y,
    seed,
    heartbeat=None
):

    set_seed(
        seed
    )

    model = ReferenceModel(
        seed=seed
    ).to(
        DEVICE
    )

    C_t = [
        torch.tensor(
            x,
            dtype=torch.long,
            device=DEVICE
        )
        for x in C
    ]

    U_t = torch.tensor(
        U,
        dtype=torch.float32,
        device=DEVICE
    )

    y_t = torch.tensor(
        y,
        dtype=torch.float32,
        device=DEVICE
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.005,
        weight_decay=5e-5
    )

    for epoch in range(
        REF_EPOCHS
    ):

        optimizer.zero_grad()

        pred = model(
            C_t,
            U_t
        )

        loss = F.mse_loss(
            pred,
            y_t
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            5.0
        )

        optimizer.step()

        model.center_embeddings_()

        model.project_mlp_orthogonal_()

        if (
            heartbeat is not None
            and
            (epoch + 1) % 50 == 0
        ):

            heartbeat()

    with torch.no_grad():

        final_loss = float(
            F.mse_loss(
                model(
                    C_t,
                    U_t
                ),
                y_t
            ).cpu()
        )

    return (
        model,
        final_loss
    )


def train_reference_model(
    C,
    U,
    y,
    seed,
    heartbeat=None
):

    best_model = None

    best_loss = np.inf

    for r in range(
        REF_RESTARTS
    ):

        model, loss = (
            train_reference_once(
                C,
                U,
                y,
                seed
                +
                1000
                * r,
                heartbeat
            )
        )

        if loss < best_loss:

            if best_model is not None:

                del best_model

            best_model = model

            best_loss = loss

        else:

            del model

        gc.collect()

    return (
        best_model,
        best_loss
    )


@torch.no_grad()
def predict_reference(
    model,
    C,
    U
):

    C_t = [
        torch.tensor(
            x,
            dtype=torch.long,
            device=DEVICE
        )
        for x in C
    ]

    U_t = torch.tensor(
        U,
        dtype=torch.float32,
        device=DEVICE
    )

    return (
        model(
            C_t,
            U_t
        )
        .cpu()
        .numpy()
    )


def extract_reference_parameters(
    model
):

    theta = [
        emb.weight
        .detach()
        .cpu()
        .numpy()
        .astype(float)
        .copy()
        for emb in model.embeddings
    ]

    theta = [
        x
        -
        x.mean(
            axis=0,
            keepdims=True
        )
        for x in theta
    ]

    gamma = (
        model.gamma
        .detach()
        .cpu()
        .numpy()
        .astype(float)
        .copy()
    )

    mu = (
        model.mu
        .detach()
        .cpu()
        .numpy()
        .astype(float)
        .copy()
    )

    W1 = (
        model.lin1.weight
        .detach()
        .cpu()
        .numpy()
        .astype(float)
        .copy()
    )

    b1 = (
        model.lin1.bias
        .detach()
        .cpu()
        .numpy()
        .astype(float)
        .copy()
    )

    W2 = (
        model.lin2.weight
        .detach()
        .cpu()
        .numpy()
        .astype(float)
        .copy()
    )

    b2 = (
        model.lin2.bias
        .detach()
        .cpu()
        .numpy()
        .astype(float)
        .copy()
    )

    weights = [
        (
            W1,
            b1
        ),
        (
            W2,
            b2
        )
    ]

    return (
        theta,
        gamma,
        mu,
        weights
    )


# ============================================================
# 14. RANK–SPECTRAL PRIOR MATCHING SECTION 2.3
# ============================================================

# The categorical predictors are sampled uniformly in Simulation 1,
# so pi_j is uniform.  H_j spans the subspace orthogonal to pi_j.

def centered_basis(
    n
):

    pi = np.full(
        n,
        1.0 / n,
        dtype=float
    )

    _, _, vh = np.linalg.svd(
        pi.reshape(1, -1),
        full_matrices=True
    )

    return vh[
        1:,
        :
    ].T.copy()


CENTER_BASES = [
    centered_basis(nj)
    for nj in N_LEVELS
]


def stable_sigmoid(
    x
):

    x = np.asarray(
        x,
        dtype=float
    )

    out = np.empty_like(
        x,
        dtype=float
    )

    positive = (
        x >= 0
    )

    out[positive] = (
        1.0
        /
        (
            1.0
            +
            np.exp(
                -x[positive]
            )
        )
    )

    exp_x = np.exp(
        x[
            ~positive
        ]
    )

    out[
        ~positive
    ] = (
        exp_x
        /
        (
            1.0
            +
            exp_x
        )
    )

    return out


def logit(
    p
):

    p = np.clip(
        p,
        1e-10,
        1.0 - 1e-10
    )

    return np.log(
        p
    ) - np.log1p(
        -p
    )


def rank_prior_probability(
    j,
    rank
):

    Kj = K_LIST[j]

    if rank < 0 or rank > Kj:
        return 0.0

    # Uniform p_j(k)>0, k=0,...,K_j.
    return 1.0 / float(
        Kj + 1
    )


def spectral_s2_from_state(
    state
):

    rank = int(
        state[
            "rank"
        ]
    )

    if rank == 0:
        return np.zeros(
            0,
            dtype=float
        )

    s2 = np.empty(
        rank,
        dtype=float
    )

    s2[0] = np.exp(
        state[
            "log_s1_sq"
        ]
    )

    if rank > 1:

        rho = stable_sigmoid(
            state[
                "eta"
            ]
        )

        for k in range(
            1,
            rank
        ):

            s2[k] = (
                rho[
                    k - 1
                ]
                *
                s2[
                    k - 1
                ]
            )

    return s2


def qr_orthonormal(
    X
):

    X = np.asarray(
        X,
        dtype=float
    )

    if X.shape[1] == 0:
        return np.zeros_like(
            X
        )

    Qm, Rm = np.linalg.qr(
        X,
        mode="reduced"
    )

    diag = np.diag(
        Rm
    )

    signs = np.sign(
        diag
    )

    signs[
        signs == 0
    ] = 1.0

    Qm = (
        Qm
        * signs
    )

    return Qm[
        :,
        :X.shape[1]
    ]


def state_orientations(
    state,
    j
):

    rank = int(
        state[
            "rank"
        ]
    )

    if rank == 0:

        return (
            np.zeros(
                (
                    N_LEVELS[j],
                    0
                ),
                dtype=float
            ),
            np.zeros(
                (
                    P_Z,
                    0
                ),
                dtype=float
            )
        )

    # Remark 2.3:
    # A_j in R^{(N_j-1) x r_j}, B_j in R^{p_z x r_j},
    # each with iid N(0,1) entries.  Their QR factors induce the
    # required Haar distributions on the admissible Stiefel sets.
    Q_u = qr_orthonormal(
        state[
            "A"
        ]
    )

    Q_v = qr_orthonormal(
        state[
            "B"
        ]
    )

    U = (
        CENTER_BASES[j]
        @
        Q_u
    )

    V = Q_v

    return (
        U,
        V
    )


def state_to_theta(
    state,
    j
):

    rank = int(
        state[
            "rank"
        ]
    )

    if rank == 0:

        return np.zeros(
            (
                N_LEVELS[j],
                P_Z
            ),
            dtype=float
        )

    U, V = state_orientations(
        state,
        j
    )

    s = np.sqrt(
        spectral_s2_from_state(
            state
        )
    )

    theta = (
        U
        @
        np.diag(
            s
        )
        @
        V.T
    )

    # For Simulation 1, pi_j is uniform, so this is exactly the
    # identification restriction pi_j^T Theta_j = 0^T up to numerical error.
    theta -= theta.mean(
        axis=0,
        keepdims=True
    )

    return theta


def initialize_rank_spectral_state(
    theta,
    j
):

    theta = (
        theta
        -
        theta.mean(
            axis=0,
            keepdims=True
        )
    )

    U, s, Vt = np.linalg.svd(
        theta,
        full_matrices=False
    )

    # Initialize at the maximal admissible rank.  Rank is subsequently
    # sampled from its posterior through the reversible-jump moves below.
    rank = K_LIST[j]

    U_r = U[
        :,
        :rank
    ]

    V_r = Vt.T[
        :,
        :rank
    ]

    Q_u = (
        CENTER_BASES[j].T
        @
        U_r
    )

    Q_u = qr_orthonormal(
        Q_u
    )

    V_r = qr_orthonormal(
        V_r
    )

    # A=Q_u and B=V_r are valid unconstrained starting values whose
    # QR factors reproduce the fitted reference orientations.
    A = Q_u.copy()
    B = V_r.copy()

    s2 = np.maximum(
        s[
            :rank
        ] ** 2,
        1e-10
    )

    s2 = np.minimum.accumulate(
        s2
    )

    eta = np.zeros(
        max(
            rank - 1,
            0
        ),
        dtype=float
    )

    if rank > 1:

        rho = np.clip(
            s2[1:]
            /
            s2[:-1],
            1e-8,
            1.0 - 1e-8
        )

        eta = logit(
            rho
        )

    return {
        "rank": rank,
        "A": A,
        "B": B,
        "log_s1_sq": float(
            np.log(
                s2[0]
            )
        ),
        "eta": eta.copy()
    }


def log_spectral_prior_transformed(
    state
):

    rank = int(
        state[
            "rank"
        ]
    )

    if rank == 0:
        return 0.0

    x = float(
        state[
            "log_s1_sq"
        ]
    )

    # s_{j1}^2 ~ Gamma(shape, rate), written on x=log(s_{j1}^2).
    # The +x Jacobian changes (shape-1)x into shape*x.
    out = (
        SINGULAR_SHAPE
        *
        x
        -
        SINGULAR_RATE
        *
        np.exp(
            x
        )
    )

    if rank > 1:

        rho = np.clip(
            stable_sigmoid(
                state[
                    "eta"
                ]
            ),
            1e-12,
            1.0 - 1e-12
        )

        # rho ~ Beta(alpha,beta), written on eta=logit(rho).
        # Including the Jacobian rho(1-rho) gives alpha*log(rho)
        # + beta*log(1-rho), up to an additive constant.
        out += float(
            np.sum(
                RATIO_ALPHA
                *
                np.log(
                    rho
                )
                +
                RATIO_BETA
                *
                np.log1p(
                    -rho
                )
            )
        )

    return float(
        out
    )


def pcn_gaussian_proposal(
    X,
    beta,
    rng
):

    if X.size == 0:
        return X.copy()

    beta = float(
        np.clip(
            beta,
            0.0,
            0.999999
        )
    )

    return (
        np.sqrt(
            1.0
            -
            beta ** 2
        )
        *
        X
        +
        beta
        *
        rng.normal(
            size=X.shape
        )
    )


def propose_local_rank_spectral_state(
    state,
    auxiliary_beta,
    spectrum_step,
    rng
):

    rank = int(
        state[
            "rank"
        ]
    )

    proposal = {
        "rank": rank,
        "A": state[
            "A"
        ].copy(),
        "B": state[
            "B"
        ].copy(),
        "log_s1_sq": state[
            "log_s1_sq"
        ],
        "eta": state[
            "eta"
        ].copy()
    }

    if rank == 0:
        return proposal

    # pCN is reversible with respect to the iid N(0,1) priors on A_j and B_j.
    # Therefore the auxiliary Gaussian prior and proposal-density terms cancel
    # from the Metropolis ratio exactly.
    proposal[
        "A"
    ] = pcn_gaussian_proposal(
        state[
            "A"
        ],
        auxiliary_beta,
        rng
    )

    proposal[
        "B"
    ] = pcn_gaussian_proposal(
        state[
            "B"
        ],
        auxiliary_beta,
        rng
    )

    proposal[
        "log_s1_sq"
    ] = float(
        state[
            "log_s1_sq"
        ]
        +
        spectrum_step
        *
        rng.normal()
    )

    if rank > 1:

        proposal[
            "eta"
        ] = (
            state[
                "eta"
            ]
            +
            spectrum_step
            *
            rng.normal(
                size=rank - 1
            )
        )

    return proposal


def propose_orientation_state(
    state,
    auxiliary_beta,
    update_A,
    rng
):

    """One pCN orientation update preserving the N(0,1) auxiliary prior."""

    rank = int(
        state[
            "rank"
        ]
    )

    proposal = {
        "rank": rank,
        "A": state[
            "A"
        ].copy(),
        "B": state[
            "B"
        ].copy(),
        "log_s1_sq": state[
            "log_s1_sq"
        ],
        "eta": state[
            "eta"
        ].copy()
    }

    if rank == 0:
        return proposal

    key = (
        "A"
        if update_A
        else "B"
    )

    proposal[
        key
    ] = pcn_gaussian_proposal(
        state[
            key
        ],
        auxiliary_beta,
        rng
    )

    return proposal


def propose_spectrum_coordinate_state(
    state,
    spectrum_step,
    rng
):

    """Random-scan update of one transformed spectral coordinate."""

    rank = int(
        state[
            "rank"
        ]
    )

    proposal = {
        "rank": rank,
        "A": state[
            "A"
        ].copy(),
        "B": state[
            "B"
        ].copy(),
        "log_s1_sq": state[
            "log_s1_sq"
        ],
        "eta": state[
            "eta"
        ].copy()
    }

    if rank == 0:
        return proposal

    # Coordinate 0 is log(s_1^2); coordinates 1,...,r-1 are logit(rho).
    coordinate = int(
        rng.integers(
            0,
            rank
        )
    )

    if coordinate == 0:

        proposal[
            "log_s1_sq"
        ] = float(
            state[
                "log_s1_sq"
            ]
            +
            spectrum_step
            *
            rng.normal()
        )

    else:

        proposal[
            "eta"
        ][
            coordinate - 1
        ] += (
            spectrum_step
            *
            rng.normal()
        )

    return proposal


def effective_sample_size(x):

    """Simple positive-sequence ESS diagnostic for a one-dimensional chain."""

    x = np.asarray(
        x,
        dtype=float
    )

    n = int(
        x.size
    )

    if n < 4:
        return float(n)

    x = x - np.mean(x)

    var = float(
        np.dot(x, x)
        /
        n
    )

    if (
        not np.isfinite(var)
        or
        var <= 1e-16
    ):
        return float(n)

    rho_sum = 0.0

    for lag in range(
        1,
        n
    ):

        acov = float(
            np.dot(
                x[:-lag],
                x[lag:]
            )
            /
            (
                n - lag
            )
        )

        rho = acov / var

        if (
            not np.isfinite(rho)
            or
            rho <= 0.0
        ):
            break

        rho_sum += rho

    return float(
        max(
            1.0,
            min(
                float(n),
                n
                /
                (
                    1.0
                    +
                    2.0
                    *
                    rho_sum
                )
            )
        )
    )


def birth_death_probabilities(
    rank,
    K
):

    if rank == 0:
        return 1.0, 0.0

    if rank == K:
        return 0.0, 1.0

    return 0.5, 0.5


def propose_rank_move(
    state,
    j,
    rng
):

    rank = int(
        state[
            "rank"
        ]
    )

    K = K_LIST[j]

    p_birth, p_death = birth_death_probabilities(
        rank,
        K
    )

    if p_birth > 0 and p_death > 0:

        do_birth = (
            rng.uniform()
            <
            p_birth
        )

    else:

        do_birth = (
            p_birth > 0
        )

    if do_birth:

        new_rank = rank + 1

        # New auxiliary columns are proposed exactly from their N(0,1)
        # priors, matching Remark 2.3.
        a_new = rng.normal(
            size=(
                N_LEVELS[j] - 1,
                1
            )
        )

        b_new = rng.normal(
            size=(
                P_Z,
                1
            )
        )

        A = np.column_stack(
            [
                state[
                    "A"
                ],
                a_new
            ]
        )

        B = np.column_stack(
            [
                state[
                    "B"
                ],
                b_new
            ]
        )

        if rank == 0:

            s1_sq = rng.gamma(
                shape=SINGULAR_SHAPE,
                scale=1.0
                /
                SINGULAR_RATE
            )

            log_s1_sq = float(
                np.log(
                    max(
                        s1_sq,
                        1e-12
                    )
                )
            )

            eta = np.zeros(
                0,
                dtype=float
            )

        else:

            rho_new = rng.beta(
                RATIO_ALPHA,
                RATIO_BETA
            )

            log_s1_sq = state[
                "log_s1_sq"
            ]

            eta = np.concatenate(
                [
                    state[
                        "eta"
                    ],
                    np.array(
                        [
                            logit(
                                rho_new
                            )
                        ],
                        dtype=float
                    )
                ]
            )

        proposal = {
            "rank": new_rank,
            "A": A,
            "B": B,
            "log_s1_sq": log_s1_sq,
            "eta": eta
        }

        _, reverse_death = birth_death_probabilities(
            new_rank,
            K
        )

        log_hastings = (
            np.log(
                rank_prior_probability(
                    j,
                    new_rank
                )
            )
            -
            np.log(
                rank_prior_probability(
                    j,
                    rank
                )
            )
            +
            np.log(
                reverse_death
            )
            -
            np.log(
                p_birth
            )
        )

    else:

        new_rank = rank - 1

        if new_rank == 0:

            proposal = {
                "rank": 0,
                "A": np.zeros(
                    (
                        N_LEVELS[j] - 1,
                        0
                    ),
                    dtype=float
                ),
                "B": np.zeros(
                    (
                        P_Z,
                        0
                    ),
                    dtype=float
                ),
                "log_s1_sq": None,
                "eta": np.zeros(
                    0,
                    dtype=float
                )
            }

        else:

            proposal = {
                "rank": new_rank,
                "A": state[
                    "A"
                ][
                    :,
                    :new_rank
                ].copy(),
                "B": state[
                    "B"
                ][
                    :,
                    :new_rank
                ].copy(),
                "log_s1_sq": state[
                    "log_s1_sq"
                ],
                "eta": state[
                    "eta"
                ][
                    :max(
                        new_rank - 1,
                        0
                    )
                ].copy()
            }

        reverse_birth, _ = birth_death_probabilities(
            new_rank,
            K
        )

        log_hastings = (
            np.log(
                rank_prior_probability(
                    j,
                    new_rank
                )
            )
            -
            np.log(
                rank_prior_probability(
                    j,
                    rank
                )
            )
            +
            np.log(
                reverse_birth
            )
            -
            np.log(
                p_death
            )
        )

    # Birth variables are proposed from their exact conditional priors.
    # Their prior/proposal factors cancel in the reversible-jump ratio;
    # only the likelihood, discrete-rank prior, and boundary move
    # probabilities remain.
    return proposal, float(
        log_hastings
    )


def log_gamma_prior(
    gamma
):

    return float(
        -0.5
        *
        np.sum(
            gamma ** 2
        )
        /
        TAU_GAMMA ** 2
    )


def log_mu_prior(
    mu
):

    return float(
        -0.5
        *
        np.sum(
            mu ** 2
        )
        /
        TAU_MU ** 2
    )


# ============================================================
# 15. LIKELIHOOD
# ============================================================

def gaussian_loglik(
    y,
    yhat,
    sigma2
):

    residual = (
        y
        -
        yhat
    )

    return float(
        -0.5
        * y.size
        * np.log(
            2.0
            * np.pi
            * sigma2
        )
        -
        0.5
        * np.sum(
            residual ** 2
        )
        / sigma2
    )


def latent_numpy(
    C,
    U,
    theta,
    gamma,
    mu
):

    n = len(
        C[
            0
        ]
    )

    z = np.tile(
        mu,
        (
            n,
            1
        )
    )

    z += (
        U
        @ gamma
    )

    for j in range(M):

        z += (
            theta[j][
                C[j],
                :
            ]
        )

    return z


# ============================================================
# 16. RANK–SPECTRAL METROPOLIS-WITHIN-GIBBS MCMC
# ============================================================

def blocked_mcmc(
    C,
    U,
    y,
    theta_init,
    gamma_init,
    mu_init,
    fixed_mlp_weights,
    sigma2_init,
    seed,
    theta_true=None,
    heartbeat=None
):

    rng = np.random.default_rng(
        seed
    )

    states = [
        initialize_rank_spectral_state(
            theta_init[j],
            j
        )
        for j in range(M)
    ]

    theta = [
        state_to_theta(
            states[j],
            j
        )
        for j in range(M)
    ]

    gamma = gamma_init.copy()
    mu = mu_init.copy()

    sigma2 = float(
        max(
            sigma2_init,
            1e-6
        )
    )

    z_cur = latent_numpy(
        C,
        U,
        theta,
        gamma,
        mu
    )

    yhat_cur = mlp_forward_numpy(
        z_cur,
        fixed_mlp_weights
    )

    ll_cur = gaussian_loglik(
        y,
        yhat_cur,
        sigma2
    )

    spectral_prior_cur = [
        log_spectral_prior_transformed(
            states[j]
        )
        for j in range(M)
    ]

    gamma_prior_cur = log_gamma_prior(
        gamma
    )

    mu_prior_cur = log_mu_prior(
        mu
    )

    n_obs = len(
        C[
            0
        ]
    )

    n_scale = np.sqrt(
        MCMC_REFERENCE_N
        /
        float(
            n_obs
        )
    )

    auxiliary_betas = np.full(
        M,
        AUXILIARY_INITIAL_BETA,
        dtype=float
    )

    spectrum_steps = np.full(
        M,
        SPECTRUM_INITIAL_STEP,
        dtype=float
    )

    gamma_step = float(
        GAMMA_INITIAL_STEP
        *
        n_scale
    )

    mu_step = float(
        MU_INITIAL_STEP
        *
        n_scale
    )

    # Separate adaptation and post-burn-in diagnostics for orientation
    # and spectral blocks.  This is intentionally more informative than
    # a single combined "Theta acceptance" statistic.
    orientation_window_accept = np.zeros(M)
    orientation_window_trials = np.zeros(M)
    spectrum_window_accept = np.zeros(M)
    spectrum_window_trials = np.zeros(M)

    gamma_window_accept = 0
    gamma_window_trials = 0
    mu_window_accept = 0
    mu_window_trials = 0

    orientation_post_accept = np.zeros(M)
    orientation_post_trials = np.zeros(M)
    spectrum_post_accept = np.zeros(M)
    spectrum_post_trials = np.zeros(M)
    rank_post_accept = np.zeros(M)
    rank_post_trials = np.zeros(M)

    gamma_post_accept = 0
    gamma_post_trials = 0
    mu_post_accept = 0
    mu_post_trials = 0

    singular_draws = [
        []
        for _ in range(M)
    ]

    rank_draws = [
        []
        for _ in range(M)
    ]

    sigma_draws = []
    theta_error_draws = []

    adapt_stop = int(
        MCMC_ADAPT_FRACTION
        *
        MCMC_BURN
    )

    adaptation_round = 0

    def evaluate_theta_proposal(j, proposal_state):

        theta_prop = state_to_theta(
            proposal_state,
            j
        )

        delta_latent = (
            theta_prop[
                C[j],
                :
            ]
            -
            theta[j][
                C[j],
                :
            ]
        )

        z_prop = (
            z_cur
            +
            delta_latent
        )

        yhat_prop = mlp_forward_numpy(
            z_prop,
            fixed_mlp_weights
        )

        ll_prop = gaussian_loglik(
            y,
            yhat_prop,
            sigma2
        )

        return (
            theta_prop,
            z_prop,
            yhat_prop,
            ll_prop
        )

    for it in range(
        MCMC_ITERS
    ):

        # ====================================================
        # A. Predictor-specific rank--spectral updates
        # ====================================================

        for j in range(M):

            current_rank = int(
                states[j][
                    "rank"
                ]
            )

            if current_rank > 0:

                # --------------------------------------------
                # A1. Orientation block: update A_j or B_j only
                # --------------------------------------------

                proposal_state = propose_orientation_state(
                    states[j],
                    auxiliary_betas[j],
                    update_A=((it + j) % 2 == 0),
                    rng=rng
                )

                (
                    theta_prop,
                    z_prop,
                    yhat_prop,
                    ll_prop
                ) = evaluate_theta_proposal(
                    j,
                    proposal_state
                )

                # pCN is reversible with respect to the Gaussian prior.
                log_ratio = (
                    ll_prop
                    -
                    ll_cur
                )

                accepted = (
                    np.log(
                        rng.uniform()
                    )
                    <
                    log_ratio
                )

                if it < adapt_stop:
                    orientation_window_trials[j] += 1

                if it >= MCMC_BURN:
                    orientation_post_trials[j] += 1

                if accepted:

                    states[j] = proposal_state
                    theta[j] = theta_prop
                    z_cur = z_prop
                    yhat_cur = yhat_prop
                    ll_cur = ll_prop

                    if it < adapt_stop:
                        orientation_window_accept[j] += 1

                    if it >= MCMC_BURN:
                        orientation_post_accept[j] += 1

                # --------------------------------------------
                # A2. Spectral block: update one coordinate only
                # --------------------------------------------

                proposal_state = propose_spectrum_coordinate_state(
                    states[j],
                    spectrum_steps[j],
                    rng
                )

                (
                    theta_prop,
                    z_prop,
                    yhat_prop,
                    ll_prop
                ) = evaluate_theta_proposal(
                    j,
                    proposal_state
                )

                spectral_prior_prop = log_spectral_prior_transformed(
                    proposal_state
                )

                log_ratio = (
                    ll_prop
                    +
                    spectral_prior_prop
                    -
                    ll_cur
                    -
                    spectral_prior_cur[j]
                )

                accepted = (
                    np.log(
                        rng.uniform()
                    )
                    <
                    log_ratio
                )

                if it < adapt_stop:
                    spectrum_window_trials[j] += 1

                if it >= MCMC_BURN:
                    spectrum_post_trials[j] += 1

                if accepted:

                    states[j] = proposal_state
                    theta[j] = theta_prop
                    z_cur = z_prop
                    yhat_cur = yhat_prop
                    ll_cur = ll_prop
                    spectral_prior_cur[j] = spectral_prior_prop

                    if it < adapt_stop:
                        spectrum_window_accept[j] += 1

                    if it >= MCMC_BURN:
                        spectrum_post_accept[j] += 1

            # -----------------------------------------------
            # A3. Adjacent reversible-jump rank move
            # -----------------------------------------------

            if (
                (it + 1)
                %
                RANK_MOVE_EVERY
                ==
                0
            ):

                proposal_state, log_hastings = propose_rank_move(
                    states[j],
                    j,
                    rng
                )

                (
                    theta_prop,
                    z_prop,
                    yhat_prop,
                    ll_prop
                ) = evaluate_theta_proposal(
                    j,
                    proposal_state
                )

                log_ratio = (
                    ll_prop
                    -
                    ll_cur
                    +
                    log_hastings
                )

                rank_accepted = (
                    np.log(
                        rng.uniform()
                    )
                    <
                    log_ratio
                )

                if it >= MCMC_BURN:
                    rank_post_trials[j] += 1

                if rank_accepted:

                    states[j] = proposal_state
                    theta[j] = theta_prop
                    z_cur = z_prop
                    yhat_cur = yhat_prop
                    ll_cur = ll_prop
                    spectral_prior_cur[j] = (
                        log_spectral_prior_transformed(
                            proposal_state
                        )
                    )

                    if it >= MCMC_BURN:
                        rank_post_accept[j] += 1

        # ====================================================
        # B. Gamma block
        # ====================================================

        gamma_prop = (
            gamma
            +
            gamma_step
            *
            rng.normal(
                size=gamma.shape
            )
        )

        z_prop = (
            z_cur
            +
            U
            @
            (
                gamma_prop
                -
                gamma
            )
        )

        yhat_prop = mlp_forward_numpy(
            z_prop,
            fixed_mlp_weights
        )

        ll_prop = gaussian_loglik(
            y,
            yhat_prop,
            sigma2
        )

        gamma_prior_prop = log_gamma_prior(
            gamma_prop
        )

        log_ratio = (
            ll_prop
            +
            gamma_prior_prop
            -
            ll_cur
            -
            gamma_prior_cur
        )

        gamma_accepted = (
            np.log(
                rng.uniform()
            )
            <
            log_ratio
        )

        if it < adapt_stop:
            gamma_window_trials += 1

        if it >= MCMC_BURN:
            gamma_post_trials += 1

        if gamma_accepted:

            gamma = gamma_prop
            z_cur = z_prop
            yhat_cur = yhat_prop
            ll_cur = ll_prop
            gamma_prior_cur = gamma_prior_prop

            if it < adapt_stop:
                gamma_window_accept += 1

            if it >= MCMC_BURN:
                gamma_post_accept += 1

        # ====================================================
        # C. mu block
        # ====================================================

        mu_prop = (
            mu
            +
            mu_step
            *
            rng.normal(
                size=mu.shape
            )
        )

        z_prop = (
            z_cur
            +
            (
                mu_prop
                -
                mu
            )
        )

        yhat_prop = mlp_forward_numpy(
            z_prop,
            fixed_mlp_weights
        )

        ll_prop = gaussian_loglik(
            y,
            yhat_prop,
            sigma2
        )

        mu_prior_prop = log_mu_prior(
            mu_prop
        )

        log_ratio = (
            ll_prop
            +
            mu_prior_prop
            -
            ll_cur
            -
            mu_prior_cur
        )

        mu_accepted = (
            np.log(
                rng.uniform()
            )
            <
            log_ratio
        )

        if it < adapt_stop:
            mu_window_trials += 1

        if it >= MCMC_BURN:
            mu_post_trials += 1

        if mu_accepted:

            mu = mu_prop
            z_cur = z_prop
            yhat_cur = yhat_prop
            ll_cur = ll_prop
            mu_prior_cur = mu_prior_prop

            if it < adapt_stop:
                mu_window_accept += 1

            if it >= MCMC_BURN:
                mu_post_accept += 1

        # ====================================================
        # D. Exact Gibbs update for sigma^2
        # ====================================================

        residual = (
            y
            -
            yhat_cur
        )

        shape_post = (
            SIGMA_A0
            +
            0.5
            *
            y.size
        )

        rate_post = (
            SIGMA_B0
            +
            0.5
            *
            np.sum(
                residual ** 2
            )
        )

        sigma2 = (
            1.0
            /
            rng.gamma(
                shape=shape_post,
                scale=1.0
                /
                rate_post
            )
        )

        ll_cur = gaussian_loglik(
            y,
            yhat_cur,
            sigma2
        )

        # ====================================================
        # E. Separate proposal adaptation
        # ====================================================

        if (
            it < adapt_stop
            and
            (it + 1)
            %
            MCMC_ADAPT_EVERY
            ==
            0
        ):

            adaptation_round += 1

            adaptive_gain = (
                MCMC_ADAPT_GAIN
                /
                np.sqrt(
                    adaptation_round
                )
            )

            for j in range(M):

                if orientation_window_trials[j] > 0:

                    orientation_rate = (
                        orientation_window_accept[j]
                        /
                        orientation_window_trials[j]
                    )

                    auxiliary_betas[j] = float(
                        np.clip(
                            auxiliary_betas[j]
                            *
                            np.exp(
                                adaptive_gain
                                *
                                (
                                    orientation_rate
                                    -
                                    MCMC_TARGET_ORIENTATION
                                )
                            ),
                            AUXILIARY_BETA_MIN,
                            AUXILIARY_BETA_MAX
                        )
                    )

                if spectrum_window_trials[j] > 0:

                    spectrum_rate = (
                        spectrum_window_accept[j]
                        /
                        spectrum_window_trials[j]
                    )

                    spectrum_steps[j] = float(
                        np.clip(
                            spectrum_steps[j]
                            *
                            np.exp(
                                adaptive_gain
                                *
                                (
                                    spectrum_rate
                                    -
                                    MCMC_TARGET_SPECTRUM
                                )
                            ),
                            SPECTRUM_STEP_MIN,
                            SPECTRUM_STEP_MAX
                        )
                    )

            if gamma_window_trials > 0:

                gamma_rate = (
                    gamma_window_accept
                    /
                    gamma_window_trials
                )

                gamma_step = float(
                    np.clip(
                        gamma_step
                        *
                        np.exp(
                            adaptive_gain
                            *
                            (
                                gamma_rate
                                -
                                MCMC_TARGET_GAMMA
                            )
                        ),
                        GAMMA_STEP_MIN,
                        GAMMA_STEP_MAX
                    )
                )

            if mu_window_trials > 0:

                mu_rate = (
                    mu_window_accept
                    /
                    mu_window_trials
                )

                mu_step = float(
                    np.clip(
                        mu_step
                        *
                        np.exp(
                            adaptive_gain
                            *
                            (
                                mu_rate
                                -
                                MCMC_TARGET_MU
                            )
                        ),
                        MU_STEP_MIN,
                        MU_STEP_MAX
                    )
                )

            orientation_window_accept[:] = 0
            orientation_window_trials[:] = 0
            spectrum_window_accept[:] = 0
            spectrum_window_trials[:] = 0
            gamma_window_accept = 0
            gamma_window_trials = 0
            mu_window_accept = 0
            mu_window_trials = 0

        # ====================================================
        # F. Save posterior draws
        # ====================================================

        if (
            it >= MCMC_BURN
            and
            (
                it
                -
                MCMC_BURN
            )
            %
            MCMC_THIN
            ==
            0
        ):

            current_theta_error = 0.0

            for j in range(M):

                rank = int(
                    states[j][
                        "rank"
                    ]
                )

                s = np.zeros(
                    K_LIST[j],
                    dtype=float
                )

                if rank > 0:

                    s[
                        :rank
                    ] = np.sqrt(
                        spectral_s2_from_state(
                            states[j]
                        )
                    )

                singular_draws[j].append(
                    s
                )

                rank_draws[j].append(
                    rank
                )

                if theta_true is not None:

                    current_theta_error += float(
                        np.sum(
                            (
                                theta[j]
                                -
                                theta_true[j]
                            ) ** 2
                        )
                    )

            if theta_true is not None:

                theta_error_draws.append(
                    current_theta_error
                )

            sigma_draws.append(
                sigma2
            )

        if (
            heartbeat is not None
            and
            (it + 1) % 50 == 0
        ):

            heartbeat()

    singular_draws = [
        np.asarray(
            x,
            dtype=float
        )
        for x in singular_draws
    ]

    rank_draws = [
        np.asarray(
            x,
            dtype=int
        )
        for x in rank_draws
    ]

    if any(
        len(x) == 0
        for x in singular_draws
    ):

        raise RuntimeError(
            "No posterior singular-value draws were retained."
        )

    mean_s2 = [
        np.mean(
            x ** 2,
            axis=0
        )
        for x in singular_draws
    ]

    posterior_mean_rank = np.array(
        [
            np.mean(x)
            for x in rank_draws
        ],
        dtype=float
    )

    full_rank_probability = np.array(
        [
            np.mean(
                rank_draws[j]
                ==
                K_LIST[j]
            )
            for j in range(M)
        ],
        dtype=float
    )

    orientation_post_acceptance = (
        orientation_post_accept
        /
        np.maximum(
            orientation_post_trials,
            1
        )
    )

    spectrum_post_acceptance = (
        spectrum_post_accept
        /
        np.maximum(
            spectrum_post_trials,
            1
        )
    )

    # Retain the old key for downstream compatibility; it is now the
    # average of the two separately monitored rank--spectral blocks.
    theta_post_acceptance = (
        0.5
        *
        (
            orientation_post_acceptance
            +
            spectrum_post_acceptance
        )
    )

    rank_post_acceptance = (
        rank_post_accept
        /
        np.maximum(
            rank_post_trials,
            1
        )
    )

    gamma_post_acceptance = float(
        gamma_post_accept
        /
        max(
            gamma_post_trials,
            1
        )
    )

    mu_post_acceptance = float(
        mu_post_accept
        /
        max(
            mu_post_trials,
            1
        )
    )

    gamma_mu_mean_acceptance = float(
        0.5
        *
        (
            gamma_post_acceptance
            +
            mu_post_acceptance
        )
    )

    posterior_rn = np.nan

    if len(theta_error_draws) > 0:

        posterior_rn = float(
            np.sqrt(
                np.mean(
                    theta_error_draws
                )
            )
        )

    # ESS is evaluated for every nontrivial singular-value tail and for
    # the rank chain.  These are diagnostics only and do not enter S(d).
    tail_ess_values = []
    rank_ess_values = []

    for j in range(M):

        draws = singular_draws[j]

        for d in range(
            1,
            K_LIST[j]
        ):

            tail_chain = np.sum(
                draws[
                    :,
                    d:
                ] ** 2,
                axis=1
            )

            tail_ess_values.append(
                effective_sample_size(
                    tail_chain
                )
            )

        rank_ess_values.append(
            effective_sample_size(
                rank_draws[j]
            )
        )

    return {
        "singular_draws": singular_draws,
        "rank_draws": rank_draws,
        "mean_s2": mean_s2,
        "posterior_mean_rank": posterior_mean_rank,
        "full_rank_probability": full_rank_probability,
        "posterior_rn": posterior_rn,
        "sigma_draws": np.asarray(
            sigma_draws
        ),
        "theta_acceptance": theta_post_acceptance,
        "orientation_acceptance": orientation_post_acceptance,
        "spectrum_acceptance": spectrum_post_acceptance,
        "rank_acceptance": rank_post_acceptance,
        "gamma_acceptance": gamma_post_acceptance,
        "mu_acceptance": mu_post_acceptance,
        "gm_acceptance": gamma_mu_mean_acceptance,
        "median_tail_ess": float(
            np.median(
                tail_ess_values
            )
        ),
        "min_tail_ess": float(
            np.min(
                tail_ess_values
            )
        ),
        "median_rank_ess": float(
            np.median(
                rank_ess_values
            )
        ),
        "auxiliary_betas": auxiliary_betas,
        "spectrum_steps": spectrum_steps,
        "gamma_step": gamma_step,
        "mu_step": mu_step
    }


# ============================================================
# 17. STRUCTURAL LOSS
# ============================================================

def tail_loss_arrays(
    mean_s2
):

    losses = []

    for s2 in mean_s2:

        k = len(
            s2
        )

        loss_j = np.array(
            [
                np.sum(
                    s2[
                        d:
                    ]
                )
                for d in range(
                    1,
                    k + 1
                )
            ],
            dtype=float
        )

        losses.append(
            loss_j
        )

    return losses


def allocation_loss(
    d,
    loss_arrays
):

    return float(
        sum(
            loss_arrays[j][
                d[j] - 1
            ]
            for j in range(
                len(
                    d
                )
            )
        )
    )


def embedding_cost(
    d
):

    return int(
        sum(
            N_LEVELS[j]
            *
            d[j]
            for j in range(M)
        )
    )


# ============================================================
# 18. MULTIPLE-CHOICE KNAPSACK DYNAMIC PROGRAMMING
# ============================================================

def dp_allocations(
    n_levels,
    k_list,
    loss_arrays,
    budgets
):

    m = len(
        n_levels
    )

    bmax = max(
        budgets
    )

    dp = np.full(
        (
            m + 1,
            bmax + 1
        ),
        np.inf
    )

    parent_d = np.full(
        (
            m + 1,
            bmax + 1
        ),
        -1,
        dtype=int
    )

    parent_b = np.full(
        (
            m + 1,
            bmax + 1
        ),
        -1,
        dtype=int
    )

    dp[
        0,
        0
    ] = 0.0


    for j in range(
        1,
        m + 1
    ):

        Nj = n_levels[
            j - 1
        ]

        Kj = k_list[
            j - 1
        ]

        for b in range(
            bmax + 1
        ):

            for d in range(
                1,
                Kj + 1
            ):

                cost = (
                    Nj
                    *
                    d
                )

                if (
                    b
                    >=
                    cost
                    and
                    np.isfinite(
                        dp[
                            j - 1,
                            b - cost
                        ]
                    )
                ):

                    value = (
                        dp[
                            j - 1,
                            b - cost
                        ]
                        +
                        loss_arrays[
                            j - 1
                        ][
                            d - 1
                        ]
                    )

                    if (
                        value
                        <
                        dp[
                            j,
                            b
                        ]
                    ):

                        dp[
                            j,
                            b
                        ] = value

                        parent_d[
                            j,
                            b
                        ] = d

                        parent_b[
                            j,
                            b
                        ] = (
                            b
                            -
                            cost
                        )


    out = {}


    for B in budgets:

        feasible_costs = np.arange(
            B + 1
        )

        values = dp[
            m,
            feasible_costs
        ]

        finite_mask = np.isfinite(
            values
        )

        if not np.any(
            finite_mask
        ):

            raise RuntimeError(
                f"No feasible allocation for budget {B}."
            )

        feasible_indices = feasible_costs[
            finite_mask
        ]

        finite_values = values[
            finite_mask
        ]

        best_cost = int(
            feasible_indices[
                np.argmin(
                    finite_values
                )
            ]
        )

        dvec = [
            None
        ] * m

        b = best_cost

        for j in range(
            m,
            0,
            -1
        ):

            d = int(
                parent_d[
                    j,
                    b
                ]
            )

            if d < 1:

                raise RuntimeError(
                    "DP backtracking failed."
                )

            dvec[
                j - 1
            ] = d

            b = int(
                parent_b[
                    j,
                    b
                ]
            )

        out[
            B
        ] = {

            "d":
                tuple(
                    dvec
                ),

            "loss":
                float(
                    dp[
                        m,
                        best_cost
                    ]
                ),

            "actual_cost":
                best_cost
        }

    return out


# ============================================================
# 19. REDUCED CANDIDATE MODEL
# ============================================================

class ReducedModel(
    nn.Module
):

    def __init__(
        self,
        d
    ):

        super().__init__()

        self.d = list(
            d
        )

        self.embeddings = nn.ModuleList(
            [
                nn.Embedding(
                    N_LEVELS[j],
                    self.d[j]
                )
                for j in range(M)
            ]
        )

        for emb in self.embeddings:

            nn.init.normal_(
                emb.weight,
                std=0.10
            )

        self.gamma = nn.Parameter(
            0.10
            *
            torch.randn(
                P_U,
                P_Z
            )
        )

        self.mu = nn.Parameter(
            torch.zeros(
                P_Z
            )
        )

        input_dim = (
            sum(
                self.d
            )
            +
            P_Z
        )

        # Section 2.4: same hidden-layer architecture as the reference
        # MLP, with only the input layer adjusted to the candidate latent
        # dimension.  The reference MLP has one tanh hidden layer of width P_Z.
        self.lin1 = nn.Linear(
            input_dim,
            RED_HIDDEN[
                0
            ]
        )

        self.lin2 = nn.Linear(
            RED_HIDDEN[
                0
            ],
            Q
        )

    def latent(
        self,
        C,
        U
    ):

        pieces = [
            self.embeddings[j](
                C[j]
            )
            for j in range(M)
        ]

        continuous = (
            self.mu.unsqueeze(
                0
            )
            +
            U
            @
            self.gamma
        )

        pieces.append(
            continuous
        )

        return torch.cat(
            pieces,
            dim=1
        )

    def forward(
        self,
        C,
        U
    ):

        z = self.latent(
            C,
            U
        )

        return self.lin2(
            torch.tanh(
                self.lin1(
                    z
                )
            )
        )

    @torch.no_grad()
    def center_embeddings_(
        self
    ):

        # Categories are uniform in Simulation 1, hence arithmetic
        # centering is exactly pi_j-weighted centering.
        for emb in self.embeddings:

            emb.weight -= (
                emb.weight.mean(
                    dim=0,
                    keepdim=True
                )
            )


# ============================================================
# 20. REDUCED-MODEL TRAINING
# ============================================================

def train_reduced_once(
    C,
    U,
    y,
    d,
    seed,
    heartbeat=None
):

    set_seed(
        seed
    )

    model = ReducedModel(
        d
    ).to(
        DEVICE
    )

    C_t = [
        torch.tensor(
            x,
            dtype=torch.long,
            device=DEVICE
        )
        for x in C
    ]

    U_t = torch.tensor(
        U,
        dtype=torch.float32,
        device=DEVICE
    )

    y_t = torch.tensor(
        y,
        dtype=torch.float32,
        device=DEVICE
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.005,
        weight_decay=RED_WEIGHT_DECAY
    )


    for epoch in range(
        RED_EPOCHS
    ):

        optimizer.zero_grad()

        pred = model(
            C_t,
            U_t
        )

        loss = F.mse_loss(
            pred,
            y_t
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            5.0
        )

        optimizer.step()

        model.center_embeddings_()

        if (
            heartbeat is not None
            and
            (epoch + 1) % 50 == 0
        ):

            heartbeat()


    with torch.no_grad():

        final_loss = float(
            F.mse_loss(
                model(
                    C_t,
                    U_t
                ),
                y_t
            ).cpu()
        )

    return (
        model,
        final_loss
    )


def train_reduced_model(
    C,
    U,
    y,
    d,
    seed,
    heartbeat=None
):

    best_model = None

    best_loss = np.inf


    for r in range(
        RED_RESTARTS
    ):

        model, loss = (
            train_reduced_once(
                C,
                U,
                y,
                d,
                seed
                +
                1000
                * r,
                heartbeat
            )
        )

        if loss < best_loss:

            if best_model is not None:

                del best_model

            best_model = model

            best_loss = loss

        else:

            del model

        gc.collect()


    return (
        best_model,
        best_loss
    )


@torch.no_grad()
def predict_reduced(
    model,
    C,
    U
):

    C_t = [
        torch.tensor(
            x,
            dtype=torch.long,
            device=DEVICE
        )
        for x in C
    ]

    U_t = torch.tensor(
        U,
        dtype=torch.float32,
        device=DEVICE
    )

    return (
        model(
            C_t,
            U_t
        )
        .cpu()
        .numpy()
    )


def mse(
    y,
    pred
):

    return float(
        np.mean(
            (
                y
                -
                pred
            ) ** 2
        )
    )


# ============================================================
# 21. POSTERIOR SPECTRAL DIAGNOSTICS
# ============================================================

def posterior_spectral_diagnostics(
    singular_draws,
    true_singular_values
):

    s2_errors = []

    tail_errors = []

    tail_widths = []


    for j in range(M):

        draws = singular_draws[
            j
        ]

        posterior_mean_s2 = np.mean(
            draws ** 2,
            axis=0
        )

        true_s2 = (
            np.asarray(
                true_singular_values[
                    j
                ]
            ) ** 2
        )

        s2_errors.extend(
            (
                posterior_mean_s2
                -
                true_s2
            ).tolist()
        )


        Kj = len(
            true_s2
        )

        for d in range(
            1,
            Kj
        ):

            posterior_tail = np.sum(
                draws[
                    :,
                    d:
                ] ** 2,
                axis=1
            )

            posterior_tail_mean = float(
                np.mean(
                    posterior_tail
                )
            )

            true_tail = float(
                np.sum(
                    true_s2[
                        d:
                    ]
                )
            )

            tail_errors.append(
                posterior_tail_mean
                -
                true_tail
            )

            lower, upper = np.quantile(
                posterior_tail,
                [
                    0.025,
                    0.975
                ]
            )

            tail_widths.append(
                upper
                -
                lower
            )


    s2_rmse = float(
        np.sqrt(
            np.mean(
                np.asarray(
                    s2_errors
                ) ** 2
            )
        )
    )

    tail_rmse = float(
        np.sqrt(
            np.mean(
                np.asarray(
                    tail_errors
                ) ** 2
            )
        )
    )

    mean_tail_width = float(
        np.mean(
            tail_widths
        )
    )


    return (
        s2_rmse,
        tail_rmse,
        mean_tail_width
    )


# ============================================================
# 22. ENUMERATION FOR STRUCTURAL SCORE ERROR
# ============================================================

def all_allocations():

    return itertools.product(
        *[
            range(
                1,
                K_LIST[j] + 1
            )
            for j in range(M)
        ]
    )


def compute_delta_max(
    posterior_losses,
    true_losses
):

    delta = 0.0


    for d in all_allocations():

        posterior_value = allocation_loss(
            d,
            posterior_losses
        )

        true_value = allocation_loss(
            d,
            true_losses
        )

        delta = max(
            delta,
            abs(
                posterior_value
                -
                true_value
            )
        )


    return float(
        delta
    )


def compute_normalized_delta(
    posterior_losses,
    true_losses
):

    delta = compute_delta_max(
        posterior_losses,
        true_losses
    )

    baseline_d = tuple(
        1
        for _ in range(M)
    )

    scale = allocation_loss(
        baseline_d,
        true_losses
    )

    return float(
        delta
        /
        max(
            scale,
            1e-12
        )
    )


# ============================================================
# 23. CHECKPOINT
# ============================================================

if (
    MODE == "2"
    and
    os.path.exists(
        CHECKPOINT_PATH
    )
):

    with open(
        CHECKPOINT_PATH,
        "rb"
    ) as f:

        STATE = pickle.load(
            f
        )

    if (
        STATE.get(
            "config_signature"
        )
        !=
        CONFIG_SIGNATURE
    ):

        raise RuntimeError(
            "The checkpoint was created using a different "
            "configuration. Start from the beginning or restore "
            "the previous configuration."
        )

    print(
        "Resumed checkpoint with",
        len(
            STATE[
                "completed"
            ]
        ),
        "completed replications."
    )

else:

    STATE = {

        "config_signature":
            CONFIG_SIGNATURE,

        "completed":
            set(),

        "replication_rows":
            [],

        "budget_rows":
            [],

        "started":
            time.time()
    }

    if MODE == "2":

        print(
            "No compatible checkpoint was found. "
            "Starting from the beginning."
        )


CHECKPOINT_CLOCK = {
    "last":
        time.time()
}


def save_tables():

    if (
        len(
            STATE[
                "replication_rows"
            ]
        )
        >
        0
    ):

        pd.DataFrame(
            STATE[
                "replication_rows"
            ]
        ).to_csv(
            RESULTS_PATH,
            index=False
        )

    if (
        len(
            STATE[
                "budget_rows"
            ]
        )
        >
        0
    ):

        pd.DataFrame(
            STATE[
                "budget_rows"
            ]
        ).to_csv(
            BUDGET_RESULTS_PATH,
            index=False
        )


def save_checkpoint(
    force=False
):

    now = time.time()

    if (
        force
        or
        now
        -
        CHECKPOINT_CLOCK[
            "last"
        ]
        >=
        CHECKPOINT_SECONDS
    ):

        temporary_path = (
            CHECKPOINT_PATH
            +
            ".tmp"
        )

        with open(
            temporary_path,
            "wb"
        ) as f:

            pickle.dump(
                STATE,
                f,
                protocol=pickle.HIGHEST_PROTOCOL
            )

        os.replace(
            temporary_path,
            CHECKPOINT_PATH
        )

        save_tables()

        CHECKPOINT_CLOCK[
            "last"
        ] = now

        print(
            "[checkpoint] saved at",
            time.strftime(
                "%H:%M:%S"
            )
        )


# ============================================================
# 24. SINGLE REPLICATION
# ============================================================

def run_replication(
    scenario,
    scenario_index,
    rep
):

    t0 = time.time()

    seed = (
        BASE_SEED
        +
        100000
        *
        scenario_index
        +
        rep
    )


    # --------------------------------------------------------
    # Truth
    # --------------------------------------------------------

    true_par = build_true_parameters(
        scenario[
            "spectral_regime"
        ]
    )


    # --------------------------------------------------------
    # Independent train / validation / test sets
    # --------------------------------------------------------

    Ctr, Utr, ytr, _ = generate_dataset(
        scenario[
            "n_train"
        ],
        true_par,
        scenario[
            "noise_sd"
        ],
        seed + 11
    )

    Cva, Uva, yva, _ = generate_dataset(
        N_VAL,
        true_par,
        scenario[
            "noise_sd"
        ],
        seed + 22
    )

    Cte, Ute, yte, _ = generate_dataset(
        N_TEST,
        true_par,
        scenario[
            "noise_sd"
        ],
        seed + 33
    )


    # --------------------------------------------------------
    # 1. Rich reference model
    # --------------------------------------------------------

    reference, reference_train_loss = (
        train_reference_model(
            Ctr,
            Utr,
            ytr,
            seed + 100,
            heartbeat=lambda:
                save_checkpoint(
                    force=False
                )
        )
    )


    reference_train_pred = predict_reference(
        reference,
        Ctr,
        Utr
    )

    reference_test_pred = predict_reference(
        reference,
        Cte,
        Ute
    )

    reference_test_mse = mse(
        yte,
        reference_test_pred
    )


    (
        theta_init,
        gamma_init,
        mu_init,
        omega_fixed
    ) = extract_reference_parameters(
        reference
    )


    sigma2_init = max(
        mse(
            ytr,
            reference_train_pred
        ),
        1e-6
    )


    # --------------------------------------------------------
    # 2. Bayesian rank–spectral inference
    # --------------------------------------------------------

    posterior = blocked_mcmc(

        Ctr,

        Utr,

        ytr,

        theta_init,

        gamma_init,

        mu_init,

        omega_fixed,

        sigma2_init,

        seed + 200,

        theta_true=true_par[
            "theta"
        ],

        heartbeat=lambda:
            save_checkpoint(
                force=False
            )
    )


    posterior_losses = tail_loss_arrays(
        posterior[
            "mean_s2"
        ]
    )


    # --------------------------------------------------------
    # 3. True population structural criterion
    # --------------------------------------------------------

    true_s2 = [
        np.asarray(
            s
        ) ** 2
        for s in true_par[
            "singular_values"
        ]
    ]

    true_losses = tail_loss_arrays(
        true_s2
    )


    # --------------------------------------------------------
    # Posterior and oracle DP allocations
    # --------------------------------------------------------

    posterior_dp = dp_allocations(
        N_LEVELS,
        K_LIST,
        posterior_losses,
        BUDGETS
    )

    oracle_dp = dp_allocations(
        N_LEVELS,
        K_LIST,
        true_losses,
        BUDGETS
    )


    # --------------------------------------------------------
    # Spectral diagnostics
    # --------------------------------------------------------

    (
        posterior_s2_rmse,
        posterior_tail_rmse,
        posterior_tail_interval_width
    ) = posterior_spectral_diagnostics(
        posterior[
            "singular_draws"
        ],
        true_par[
            "singular_values"
        ]
    )


    delta_max = compute_delta_max(
        posterior_losses,
        true_losses
    )

    normalized_delta = compute_normalized_delta(
        posterior_losses,
        true_losses
    )


    # --------------------------------------------------------
    # Refit ONLY structurally selected allocations
    # --------------------------------------------------------

    unique_allocations = sorted(
        set(
            posterior_dp[
                B
            ][
                "d"
            ]
            for B in BUDGETS
        )
    )


    candidate_cache = {}


    for candidate_index, d in enumerate(
        unique_allocations
    ):

        candidate_seed = (
            seed
            +
            10000
            +
            100
            *
            candidate_index
            +
            sum(
                (
                    j + 1
                )
                *
                d[j]
                for j in range(M)
            )
        )

        reduced, training_loss = (
            train_reduced_model(

                Ctr,

                Utr,

                ytr,

                d,

                candidate_seed,

                heartbeat=lambda:
                    save_checkpoint(
                        force=False
                    )
            )
        )


        validation_pred = predict_reduced(
            reduced,
            Cva,
            Uva
        )

        test_pred = predict_reduced(
            reduced,
            Cte,
            Ute
        )


        candidate_cache[
            d
        ] = {

            "train_mse":
                training_loss,

            "val_mse":
                mse(
                    yva,
                    validation_pred
                ),

            "test_mse":
                mse(
                    yte,
                    test_pred
                ),

            "total_parameters":
                int(
                    sum(
                        p.numel()
                        for p in reduced.parameters()
                    )
                )
        }


        del reduced

        gc.collect()


    # --------------------------------------------------------
    # 4. Budget-level structural diagnostics
    # --------------------------------------------------------

    budget_rows = []

    exact_recovery_indicators = []

    allocation_l1_errors = []

    structural_excesses = []


    for B in BUDGETS:

        d_post = posterior_dp[
            B
        ][
            "d"
        ]

        d_oracle = oracle_dp[
            B
        ][
            "d"
        ]

        candidate = candidate_cache[
            d_post
        ]


        true_structural_loss_selected = allocation_loss(
            d_post,
            true_losses
        )

        true_structural_loss_oracle = allocation_loss(
            d_oracle,
            true_losses
        )


        structural_excess = (
            true_structural_loss_selected
            -
            true_structural_loss_oracle
        )


        exact_recovery = float(
            tuple(
                d_post
            )
            ==
            tuple(
                d_oracle
            )
        )


        l1_error = float(
            np.mean(
                np.abs(
                    np.asarray(
                        d_post
                    )
                    -
                    np.asarray(
                        d_oracle
                    )
                )
            )
        )


        exact_recovery_indicators.append(
            exact_recovery
        )

        allocation_l1_errors.append(
            l1_error
        )

        structural_excesses.append(
            structural_excess
        )


        budget_rows.append(
            {

                "scenario":
                    scenario[
                        "name"
                    ],

                "spectral_regime":
                    scenario[
                        "spectral_regime"
                    ],

                "n_train":
                    scenario[
                        "n_train"
                    ],

                "rep":
                    rep,

                "B":
                    B,

                "posterior_d":
                    str(
                        tuple(
                            d_post
                        )
                    ),

                "oracle_d":
                    str(
                        tuple(
                            d_oracle
                        )
                    ),

                "actual_cost":
                    embedding_cost(
                        d_post
                    ),

                "posterior_structural_score":
                    posterior_dp[
                        B
                    ][
                        "loss"
                    ],

                "true_structural_loss":
                    true_structural_loss_selected,

                "oracle_structural_loss":
                    true_structural_loss_oracle,

                "structural_excess":
                    structural_excess,

                "exact_allocation_recovery":
                    exact_recovery,

                "allocation_l1_error":
                    l1_error,

                "training_mse":
                    candidate[
                        "train_mse"
                    ],

                "validation_mse":
                    candidate[
                        "val_mse"
                    ],

                "test_mse":
                    candidate[
                        "test_mse"
                    ]
            }
        )


    # --------------------------------------------------------
    # 5. Adaptive budget selection
    # --------------------------------------------------------

    BMAX = max(
        BUDGETS
    )

    d_bmax = posterior_dp[
        BMAX
    ][
        "d"
    ]

    max_budget_val = candidate_cache[
        d_bmax
    ][
        "val_mse"
    ]


    admissible_budgets = [
        B
        for B in sorted(
            BUDGETS
        )
        if (
            candidate_cache[
                posterior_dp[
                    B
                ][
                    "d"
                ]
            ][
                "val_mse"
            ]
            <=
            max_budget_val
            +
            ZETA_N
        )
    ]


    B_hat = min(
        admissible_budgets
    )

    d_hat = posterior_dp[
        B_hat
    ][
        "d"
    ]


    final_candidate = candidate_cache[
        d_hat
    ]

    max_candidate = candidate_cache[
        d_bmax
    ]


    # --------------------------------------------------------
    # Large-test-sample oracle budget
    # --------------------------------------------------------

    max_budget_test = candidate_cache[
        d_bmax
    ][
        "test_mse"
    ]


    test_oracle_admissible = [
        B
        for B in sorted(
            BUDGETS
        )
        if (
            candidate_cache[
                posterior_dp[
                    B
                ][
                    "d"
                ]
            ][
                "test_mse"
            ]
            <=
            max_budget_test
            +
            ZETA_N
        )
    ]


    B_test_oracle = min(
        test_oracle_admissible
    )


    budget_exact_match = float(
        B_hat
        ==
        B_test_oracle
    )


    budget_absolute_error = float(
        abs(
            B_hat
            -
            B_test_oracle
        )
    )


    # --------------------------------------------------------
    # Parameter reduction
    # --------------------------------------------------------

    rich_embedding_parameters = sum(
        N_LEVELS[j]
        *
        P_Z
        for j in range(M)
    )


    selected_embedding_parameters = embedding_cost(
        d_hat
    )


    embedding_reduction = (
        1.0
        -
        selected_embedding_parameters
        /
        rich_embedding_parameters
    )


    # --------------------------------------------------------
    # Runtime
    # --------------------------------------------------------

    runtime_seconds = (
        time.time()
        -
        t0
    )


    # --------------------------------------------------------
    # Replication-level result
    # --------------------------------------------------------

    rep_row = {

        "scenario":
            scenario[
                "name"
            ],

        "spectral_regime":
            scenario[
                "spectral_regime"
            ],

        "n_train":
            scenario[
                "n_train"
            ],

        "rep":
            rep,

        "noise_sd":
            scenario[
                "noise_sd"
            ],

        "selected_budget":
            B_hat,

        "test_oracle_budget":
            B_test_oracle,

        "budget_exact_match":
            budget_exact_match,

        "budget_absolute_error":
            budget_absolute_error,

        "selected_d":
            str(
                tuple(
                    d_hat
                )
            ),

        "selected_embedding_parameters":
            selected_embedding_parameters,

        "rich_embedding_parameters":
            rich_embedding_parameters,

        "embedding_reduction":
            embedding_reduction,

        "allocation_recovery_rate":
            float(
                np.mean(
                    exact_recovery_indicators
                )
            ),

        "allocation_mean_l1_error":
            float(
                np.mean(
                    allocation_l1_errors
                )
            ),

        "mean_structural_excess":
            float(
                np.mean(
                    structural_excesses
                )
            ),

        "delta_score_error":
            delta_max,

        "normalized_delta_score_error":
            normalized_delta,

        "posterior_s2_rmse":
            posterior_s2_rmse,

        "posterior_tail_rmse":
            posterior_tail_rmse,

        "posterior_tail_interval_width":
            posterior_tail_interval_width,

        # Posterior Frobenius contraction quantity from Theorem 2.5,
        # available in simulation because Theta_0 is known.
        "posterior_rn":
            float(
                posterior[
                    "posterior_rn"
                ]
            ),

        "reference_train_mse":
            reference_train_loss,

        "reference_test_mse":
            reference_test_mse,

        "final_validation_mse":
            final_candidate[
                "val_mse"
            ],

        "final_test_mse":
            final_candidate[
                "test_mse"
            ],

        "max_budget_test_mse":
            max_candidate[
                "test_mse"
            ],

        "test_excess_vs_max_budget":
            (
                final_candidate[
                    "test_mse"
                ]
                -
                max_candidate[
                    "test_mse"
                ]
            ),

        # ----------------------------------------------------
        # Post-burn-in MCMC diagnostics
        # ----------------------------------------------------

        "mean_orientation_acceptance":
            float(
                np.mean(
                    posterior[
                        "orientation_acceptance"
                    ]
                )
            ),

        "mean_spectrum_acceptance":
            float(
                np.mean(
                    posterior[
                        "spectrum_acceptance"
                    ]
                )
            ),

        "median_tail_ess":
            float(
                posterior[
                    "median_tail_ess"
                ]
            ),

        "min_tail_ess":
            float(
                posterior[
                    "min_tail_ess"
                ]
            ),

        "median_rank_ess":
            float(
                posterior[
                    "median_rank_ess"
                ]
            ),

        "mean_theta_acceptance":
            float(
                np.mean(
                    posterior[
                        "theta_acceptance"
                    ]
                )
            ),

        "min_theta_acceptance":
            float(
                np.min(
                    posterior[
                        "theta_acceptance"
                    ]
                )
            ),

        "max_theta_acceptance":
            float(
                np.max(
                    posterior[
                        "theta_acceptance"
                    ]
                )
            ),

        "mean_rank_acceptance":
            float(
                np.mean(
                    posterior[
                        "rank_acceptance"
                    ]
                )
            ),

        "mean_posterior_rank":
            float(
                np.mean(
                    posterior[
                        "posterior_mean_rank"
                    ]
                )
            ),

        "mean_full_rank_probability":
            float(
                np.mean(
                    posterior[
                        "full_rank_probability"
                    ]
                )
            ),

        "gamma_acceptance":
            float(
                posterior[
                    "gamma_acceptance"
                ]
            ),

        "mu_acceptance":
            float(
                posterior[
                    "mu_acceptance"
                ]
            ),

        "gamma_mu_acceptance":
            float(
                posterior[
                    "gm_acceptance"
                ]
            ),

        "final_gamma_step":
            float(
                posterior[
                    "gamma_step"
                ]
            ),

        "final_mu_step":
            float(
                posterior[
                    "mu_step"
                ]
            ),

        "n_unique_refits":
            len(
                unique_allocations
            ),

        "runtime_seconds":
            runtime_seconds
    }


    del reference

    gc.collect()


    return (
        rep_row,
        budget_rows
    )


# ============================================================
# 25. RUN SIMULATION
# ============================================================

print(
    "\n"
    "SIMULATION 1 — RANK–SPECTRAL BAYESIAN DESIGN\n"
    "---------------------------------------------"
)

print(
    "Spectral regimes:",
    [
        "clear",
        "weak"
    ]
)

print(
    "Training sizes:",
    [
        500,
        1000,
        2000
    ]
)

print(
    "N_j:",
    N_LEVELS
)

print(
    "K_j:",
    K_LIST
)

print(
    "Candidate budgets:",
    BUDGETS
)

print(
    "Minimum budget:",
    MIN_COST
)

print(
    "Maximum budget:",
    MAX_COST
)

print(
    "Budget tolerance zeta_n:",
    ZETA_N
)

print(
    "Noise SD:",
    NOISE_SD
)

print(
    "Monte Carlo replications per scenario:",
    N_REPS
)

print(
    "Pilot mode:",
    PILOT_MODE
)

print(
    "MCMC targets: orientation =",
    MCMC_TARGET_ORIENTATION,
    ", spectrum =",
    MCMC_TARGET_SPECTRUM,
    ", gamma =",
    MCMC_TARGET_GAMMA,
    ", mu =",
    MCMC_TARGET_MU
)

print(
    "---------------------------------------------\n"
)


try:

    for scenario_index, scenario in enumerate(
        SCENARIOS
    ):

        for rep in range(
            1,
            N_REPS + 1
        ):

            task_key = (
                scenario[
                    "name"
                ],
                rep
            )


            if (
                task_key
                in
                STATE[
                    "completed"
                ]
            ):

                continue


            print(
                f"\n[{scenario['name']}] "
                f"replication {rep}/{N_REPS}"
            )


            rep_row, budget_rows = run_replication(
                scenario,
                scenario_index,
                rep
            )


            STATE[
                "replication_rows"
            ].append(
                rep_row
            )


            STATE[
                "budget_rows"
            ].extend(
                budget_rows
            )


            STATE[
                "completed"
            ].add(
                task_key
            )


            save_checkpoint(
                force=True
            )


            print(
                "  B_hat =",
                rep_row[
                    "selected_budget"
                ],
                "| test-oracle B =",
                rep_row[
                    "test_oracle_budget"
                ],
                "| d =",
                rep_row[
                    "selected_d"
                ],
                "| recovery =",
                round(
                    rep_row[
                        "allocation_recovery_rate"
                    ],
                    3
                ),
                "| delta_norm =",
                round(
                    rep_row[
                        "normalized_delta_score_error"
                    ],
                    3
                ),
                "| test MSE =",
                round(
                    rep_row[
                        "final_test_mse"
                    ],
                    5
                )
            )

            print(
                "  post-burn acceptance:"
                " theta =",
                round(
                    rep_row[
                        "mean_theta_acceptance"
                    ],
                    3
                ),
                "| rank =",
                round(
                    rep_row[
                        "mean_rank_acceptance"
                    ],
                    3
                ),
                "| gamma =",
                round(
                    rep_row[
                        "gamma_acceptance"
                    ],
                    3
                ),
                "| mu =",
                round(
                    rep_row[
                        "mu_acceptance"
                    ],
                    3
                )
            )


except KeyboardInterrupt:

    print(
        "\nInterrupted by user. "
        "Saving checkpoint..."
    )

    save_checkpoint(
        force=True
    )

    raise


except Exception:

    print(
        "\nAn error occurred. "
        "Saving checkpoint..."
    )

    save_checkpoint(
        force=True
    )

    raise


# ============================================================
# 26. SAVE FINAL RESULTS
# ============================================================

rep_df = pd.DataFrame(
    STATE[
        "replication_rows"
    ]
)

budget_df = pd.DataFrame(
    STATE[
        "budget_rows"
    ]
)


rep_df.to_csv(
    RESULTS_PATH,
    index=False
)

budget_df.to_csv(
    BUDGET_RESULTS_PATH,
    index=False
)


# ============================================================
# 27. MAIN SUMMARY
# ============================================================

summary = (
    rep_df
    .groupby(
        [
            "spectral_regime",
            "n_train"
        ]
    )
    .agg(

        n_rep=(
            "rep",
            "count"
        ),

        allocation_recovery=(
            "allocation_recovery_rate",
            "mean"
        ),

        allocation_l1_error=(
            "allocation_mean_l1_error",
            "mean"
        ),

        structural_excess=(
            "mean_structural_excess",
            "mean"
        ),

        delta_score_error=(
            "delta_score_error",
            "mean"
        ),

        normalized_delta=(
            "normalized_delta_score_error",
            "mean"
        ),

        posterior_s2_rmse=(
            "posterior_s2_rmse",
            "mean"
        ),

        posterior_tail_rmse=(
            "posterior_tail_rmse",
            "mean"
        ),

        posterior_tail_width=(
            "posterior_tail_interval_width",
            "mean"
        ),

        mean_selected_budget=(
            "selected_budget",
            "mean"
        ),

        mean_test_oracle_budget=(
            "test_oracle_budget",
            "mean"
        ),

        budget_match_rate=(
            "budget_exact_match",
            "mean"
        ),

        budget_absolute_error=(
            "budget_absolute_error",
            "mean"
        ),

        embedding_reduction=(
            "embedding_reduction",
            "mean"
        ),

        final_test_mse=(
            "final_test_mse",
            "mean"
        ),

        max_budget_test_mse=(
            "max_budget_test_mse",
            "mean"
        ),

        reference_test_mse=(
            "reference_test_mse",
            "mean"
        ),

        test_excess_vs_max_budget=(
            "test_excess_vs_max_budget",
            "mean"
        ),

        mean_orientation_acceptance=(
            "mean_orientation_acceptance",
            "mean"
        ),

        mean_spectrum_acceptance=(
            "mean_spectrum_acceptance",
            "mean"
        ),

        median_tail_ess=(
            "median_tail_ess",
            "mean"
        ),

        min_tail_ess=(
            "min_tail_ess",
            "mean"
        ),

        median_rank_ess=(
            "median_rank_ess",
            "mean"
        ),

        mean_theta_acceptance=(
            "mean_theta_acceptance",
            "mean"
        ),

        mean_rank_acceptance=(
            "mean_rank_acceptance",
            "mean"
        ),

        mean_posterior_rank=(
            "mean_posterior_rank",
            "mean"
        ),

        mean_full_rank_probability=(
            "mean_full_rank_probability",
            "mean"
        ),

        gamma_acceptance=(
            "gamma_acceptance",
            "mean"
        ),

        mu_acceptance=(
            "mu_acceptance",
            "mean"
        ),

        gamma_mu_acceptance=(
            "gamma_mu_acceptance",
            "mean"
        ),

        mean_final_gamma_step=(
            "final_gamma_step",
            "mean"
        ),

        mean_final_mu_step=(
            "final_mu_step",
            "mean"
        ),

        mean_unique_refits=(
            "n_unique_refits",
            "mean"
        ),

        runtime_seconds=(
            "runtime_seconds",
            "mean"
        )
    )
    .reset_index()
)


# ============================================================
# 28. STANDARD ERRORS
# ============================================================

def standard_error(
    x
):

    x = np.asarray(
        x,
        dtype=float
    )

    x = x[
        np.isfinite(
            x
        )
    ]

    if len(
        x
    ) <= 1:

        return np.nan

    return float(
        np.std(
            x,
            ddof=1
        )
        /
        np.sqrt(
            len(
                x
            )
        )
    )


se_rows = []


for (
    spectral_regime,
    n_train
), tmp in rep_df.groupby(
    [
        "spectral_regime",
        "n_train"
    ]
):

    se_rows.append(
        {

            "spectral_regime":
                spectral_regime,

            "n_train":
                n_train,

            "allocation_recovery_se":
                standard_error(
                    tmp[
                        "allocation_recovery_rate"
                    ]
                ),

            "allocation_l1_error_se":
                standard_error(
                    tmp[
                        "allocation_mean_l1_error"
                    ]
                ),

            "normalized_delta_se":
                standard_error(
                    tmp[
                        "normalized_delta_score_error"
                    ]
                ),

            "selected_budget_se":
                standard_error(
                    tmp[
                        "selected_budget"
                    ]
                ),

            "final_test_mse_se":
                standard_error(
                    tmp[
                        "final_test_mse"
                    ]
                ),

            "theta_acceptance_se":
                standard_error(
                    tmp[
                        "mean_theta_acceptance"
                    ]
                ),

            "gamma_acceptance_se":
                standard_error(
                    tmp[
                        "gamma_acceptance"
                    ]
                ),

            "mu_acceptance_se":
                standard_error(
                    tmp[
                        "mu_acceptance"
                    ]
                )
        }
    )


se_df = pd.DataFrame(
    se_rows
)


summary = summary.merge(
    se_df,
    on=[
        "spectral_regime",
        "n_train"
    ],
    how="left"
)


summary.to_csv(
    SUMMARY_PATH,
    index=False
)


# ============================================================
# 29. FIGURES
# ============================================================

# ------------------------------------------------------------
# Figure 1:
# Allocation recovery versus n
# ------------------------------------------------------------

plt.figure(
    figsize=(
        8,
        5
    )
)

for regime in [
    "clear",
    "weak"
]:

    tmp = summary[
        summary[
            "spectral_regime"
        ]
        ==
        regime
    ].sort_values(
        "n_train"
    )

    plt.plot(
        tmp[
            "n_train"
        ],
        tmp[
            "allocation_recovery"
        ],
        marker="o",
        label=regime
    )

plt.xlabel(
    "Training sample size"
)

plt.ylabel(
    "Exact allocation recovery rate"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        ROOT,
        "sim1_recovery_vs_n.png"
    ),
    dpi=180
)

plt.close()


# ------------------------------------------------------------
# Figure 2:
# Normalized structural-score error versus n
# ------------------------------------------------------------

plt.figure(
    figsize=(
        8,
        5
    )
)

for regime in [
    "clear",
    "weak"
]:

    tmp = summary[
        summary[
            "spectral_regime"
        ]
        ==
        regime
    ].sort_values(
        "n_train"
    )

    plt.plot(
        tmp[
            "n_train"
        ],
        tmp[
            "normalized_delta"
        ],
        marker="o",
        label=regime
    )

plt.xlabel(
    "Training sample size"
)

plt.ylabel(
    "Normalized structural-score discrepancy"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        ROOT,
        "sim1_delta_vs_n.png"
    ),
    dpi=180
)

plt.close()


# ------------------------------------------------------------
# Figure 3:
# Selected budget versus n
# ------------------------------------------------------------

plt.figure(
    figsize=(
        8,
        5
    )
)

for regime in [
    "clear",
    "weak"
]:

    tmp = summary[
        summary[
            "spectral_regime"
        ]
        ==
        regime
    ].sort_values(
        "n_train"
    )

    plt.plot(
        tmp[
            "n_train"
        ],
        tmp[
            "mean_selected_budget"
        ],
        marker="o",
        label=regime
    )

plt.xlabel(
    "Training sample size"
)

plt.ylabel(
    "Mean selected budget"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        ROOT,
        "sim1_selected_budget.png"
    ),
    dpi=180
)

plt.close()


# ------------------------------------------------------------
# Figure 4:
# Selected versus large-test oracle budget
# ------------------------------------------------------------

plt.figure(
    figsize=(
        8,
        5
    )
)

x_labels = [
    f"{r}\n$n={n}$"
    for r, n in zip(
        summary[
            "spectral_regime"
        ],
        summary[
            "n_train"
        ]
    )
]

x = np.arange(
    len(
        summary
    )
)

width = 0.35

plt.bar(
    x
    -
    width / 2,
    summary[
        "mean_selected_budget"
    ],
    width,
    label="Validation-selected"
)

plt.bar(
    x
    +
    width / 2,
    summary[
        "mean_test_oracle_budget"
    ],
    width,
    label="Large-test oracle"
)

plt.xticks(
    x,
    x_labels
)

plt.ylabel(
    "Budget"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        ROOT,
        "sim1_budget_oracle_comparison.png"
    ),
    dpi=180
)

plt.close()


# ------------------------------------------------------------
# Figure 5:
# Predictive performance
# ------------------------------------------------------------

plt.figure(
    figsize=(
        9,
        5
    )
)

x = np.arange(
    len(
        summary
    )
)

width = 0.30

plt.bar(
    x
    -
    width,
    summary[
        "final_test_mse"
    ],
    width,
    label="Adaptive"
)

plt.bar(
    x,
    summary[
        "max_budget_test_mse"
    ],
    width,
    label="Maximum budget"
)

plt.bar(
    x
    +
    width,
    summary[
        "reference_test_mse"
    ],
    width,
    label="Reference model"
)

plt.xticks(
    x,
    x_labels
)

plt.ylabel(
    "Test MSE"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        ROOT,
        "sim1_predictive_performance.png"
    ),
    dpi=180
)

plt.close()


# ------------------------------------------------------------
# Figure 6:
# Post-burn-in MCMC acceptance rates
# ------------------------------------------------------------

plt.figure(
    figsize=(
        9,
        5
    )
)

x = np.arange(
    len(
        summary
    )
)

width = 0.25

plt.bar(
    x - width,
    summary[
        "mean_theta_acceptance"
    ],
    width,
    label=r"$\Theta$"
)

plt.bar(
    x,
    summary[
        "gamma_acceptance"
    ],
    width,
    label=r"$\Gamma$"
)

plt.bar(
    x + width,
    summary[
        "mu_acceptance"
    ],
    width,
    label=r"$\mu$"
)

plt.axhline(
    0.15,
    color="grey",
    linestyle="--",
    linewidth=1
)

plt.axhline(
    0.35,
    color="grey",
    linestyle="--",
    linewidth=1
)

plt.xticks(
    x,
    x_labels
)

plt.ylabel(
    "Post-burn-in acceptance rate"
)

plt.ylim(
    0,
    0.6
)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        ROOT,
        "sim1_mcmc_acceptance.png"
    ),
    dpi=180
)

plt.close()


# ============================================================
# 30. FINAL CHECKPOINT
# ============================================================

STATE[
    "finished"
] = True

save_checkpoint(
    force=True
)


# ============================================================
# 31. OUTPUT
# ============================================================

print(
    "\nSIMULATION 1 RANK–SPECTRAL BAYESIAN RUN COMPLETED."
)

print(
    "\nMain summary:\n"
)

print(
    summary.to_string(
        index=False
    )
)

print(
    "\n\nCHECK BEFORE THE FINAL 100-REPLICATION RUN:"
)

print(
    "\n1. Statistical behavior should remain essentially unchanged:"
)

print(
    "   recovery should increase with n and normalized delta should decrease."
)

print(
    "\n2. Clear spectra should continue to select substantially smaller budgets."
)

print(
    "\n3. Weak spectra should continue to retain the full or near-full budget."
)

print(
    "\n4. Adaptive test MSE should remain close to maximum-budget test MSE."
)

print(
    "\n5. POST-BURN-IN acceptance rates should preferably lie roughly in:"
)

print(
    "   0.15--0.45 for orientation/spectrum blocks and 0.15--0.35 for Gamma/mu moves."
)

print(
    "\n6. In particular, Gamma/mu acceptance at n=2000 should remain stable."
)

print(
    "\nAlso inspect rank acceptance, posterior rank probabilities, tail ESS and nonzero posterior tail widths before the final run."
)

print(
    "\nFiles saved in:"
)

print(
    ROOT
)

Simulation 1 theory-consistent: type 1 to START FROM THE BEGINNING or 2 to RESUME FROM THE LAST CHECKPOINT: 2
Mounted at /content/drive
Resumed checkpoint with 600 completed replications.

SIMULATION 1 — RANK–SPECTRAL BAYESIAN DESIGN
---------------------------------------------
Spectral regimes: ['clear', 'weak']
Training sizes: [500, 1000, 2000]
N_j: [8, 10, 12, 9]
K_j: [4, 4, 4, 4]
Candidate budgets: [39, 60, 80, 100, 120, 140, 156]
Minimum budget: 39
Maximum budget: 156
Budget tolerance zeta_n: 0.0015
Noise SD: 0.1
Monte Carlo replications per scenario: 100
Pilot mode: False
MCMC targets: orientation = 0.3 , spectrum = 0.3 , gamma = 0.25 , mu = 0.3
---------------------------------------------

[checkpoint] saved at 10:03:51

SIMULATION 1 RANK–SPECTRAL BAYESIAN RUN COMPLETED.

Main summary:

spectral_regime  n_train  n_rep  allocation_recovery  allocation_l1_error  structural_excess  delta_score_error  normalized_delta  posterior_s2_rmse  posterior_tail_rmse  posterior_tail_width  

In [4]:
import numpy as np

def compute_zeta_from_validation_losses(
    val_losses,
    alpha=0.05,
    n_boot=5000,
    seed=123
):
    """
    Parameters
    ----------
    val_losses : array, shape (n_val, n_candidates)
        Entry [i, b] is the validation loss of observation i
        for the structurally selected model corresponding to budget b.

    alpha : float
        Simultaneous error probability.

    n_boot : int
        Number of multiplier-bootstrap draws.

    Returns
    -------
    t_hat : float
        Estimated simultaneous validation uncertainty radius.

    zeta_hat : float
        Recommended tolerance: 2 * t_hat.
    """

    L = np.asarray(val_losses, dtype=float)

    if L.ndim != 2:
        raise ValueError(
            "val_losses must have shape (n_val, n_candidates)."
        )

    n_val, n_candidates = L.shape

    # Center each candidate's observation-level validation losses
    centered = L - L.mean(axis=0, keepdims=True)

    rng = np.random.default_rng(seed)

    max_deviations = np.empty(n_boot)

    for b in range(n_boot):
        # Gaussian multiplier bootstrap
        xi = rng.standard_normal(n_val)

        bootstrap_deviation = (
            xi[:, None] * centered
        ).sum(axis=0) / n_val

        max_deviations[b] = np.max(
            np.abs(bootstrap_deviation)
        )

    t_hat = np.quantile(max_deviations, 1.0 - alpha)

    # Matches the sufficient condition zeta_n >= 2 t_{n,alpha}
    zeta_hat = 2.0 * t_hat

    return t_hat, zeta_hat